# 05 · Experimento V5 · Contexto climatológico ERA5-Land

Este experimento parte de **V3 (167 características)** y añade únicamente
información climatológica derivada de ERA5-Land.

La diferencia metodológica importante es que la climatología se vuelve a
calcular dentro de cada fold usando exclusivamente su subperíodo de
entrenamiento. Así se evita utilizar años futuros para transformar la
validación del fold.

**Todavía no se usa 2018–2021, prueba temporal ni holdout espacial.**

## 0. Dependencias

In [1]:
%pip install -q scikit-learn xgboost joblib matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [2]:
from pathlib import Path
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Entorno listo.")

Entorno listo.


## 2. Localizar proyecto

In [3]:
candidates = [
    Path.cwd(),
    Path.cwd() / "rain-threat-classifier",
    Path("/content/rain-threat-classifier"),
    Path("/content/drive/MyDrive/rain-threat-classifier"),
]

PROJECT_DIR = next(
    (
        p for p in candidates
        if (p / "resultados_completo" / "dataset_modelo_mensual_v3.csv").exists()
        and (p / "04_climatologia_era5land.py").exists()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró el proyecto con dataset V3 y "
        "04_climatologia_era5land.py. Define PROJECT_DIR manualmente."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from importlib import reload
import importlib
clim_module = importlib.import_module("04_climatologia_era5land")
clim_module = reload(clim_module)

from importlib import import_module
clim = import_module("04_climatologia_era5land")

DATA_DIR = PROJECT_DIR / "resultados_completo"
OUTPUT_DIR = PROJECT_DIR / "resultados_experimentos" / "v5_climatologia_cv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)

PROJECT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier


## 3. Carga de V3 y datos mensuales ERA5-Land

In [4]:
DATASET_PATH = DATA_DIR / "dataset_modelo_mensual_v3.csv"
FEATURES_PATH = DATA_DIR / "columnas_modelo_v3.txt"
MONTHLY_PATH = DATA_DIR / "indicadores_mensuales_todas_zonas.csv"

df = pd.read_csv(DATASET_PATH)
df["period_start"] = pd.to_datetime(df["period_start"])
df["target_period_start"] = pd.to_datetime(df["target_period_start"])

monthly = pd.read_csv(MONTHLY_PATH)
monthly["period_start"] = pd.to_datetime(monthly["period_start"])

features_v3 = [
    line.strip()
    for line in FEATURES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

features_v5 = features_v3 + clim.CLIMATE_FEATURE_NAMES

print("Dataset V3:", df.shape)
print("Features V3:", len(features_v3))
print("Features climatológicas V5:", len(clim.CLIMATE_FEATURE_NAMES))
print("Features totales V5:", len(features_v5))

assert len(features_v3) == 167
assert len(clim.CLIMATE_FEATURE_NAMES) == 25
assert len(features_v5) == 192
assert "target_amenaza" not in features_v5
assert "target_rx5day_mm" not in features_v5

Dataset V3: (6120, 209)
Features V3: 167
Features climatológicas V5: 25
Features totales V5: 192


## 4. Conjunto de entrenamiento y clases

Solo se usa `split = entrenamiento`. Las demás particiones permanecen cerradas.

In [5]:
train_df = df[df["split"] == "entrenamiento"].copy()

label_encoder = LabelEncoder()
label_encoder.fit(train_df["target_amenaza"])
CLASS_NAMES = list(label_encoder.classes_)

print("Entrenamiento:", train_df.shape)
print("Clases:", dict(enumerate(CLASS_NAMES)))
print("Periodo objetivo:",
      train_df["target_period_start"].min(),
      "→",
      train_df["target_period_start"].max())

Entrenamiento: (3744, 209)
Clases: {0: 'Alta', 1: 'Baja', 2: 'Media'}
Periodo objetivo: 1992-01-01 00:00:00 → 2017-12-01 00:00:00


## 5. Folds temporales

In [6]:
TEMPORAL_FOLDS = [
    ("F1", "2004-12-01", "2005-01-01", "2007-12-01"),
    ("F2", "2007-12-01", "2008-01-01", "2010-12-01"),
    ("F3", "2010-12-01", "2011-01-01", "2013-12-01"),
    ("F4", "2013-12-01", "2014-01-01", "2017-12-01"),
]

fold_descriptions = []

for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
    tr = train_df[
        train_df["target_period_start"] <= pd.Timestamp(train_end)
    ]
    va = train_df[
        train_df["target_period_start"].between(
            pd.Timestamp(val_start),
            pd.Timestamp(val_end),
        )
    ]

    fold_descriptions.append({
        "fold": name,
        "train_filas": len(tr),
        "val_filas": len(va),
        "train_target_hasta": train_end,
        "val_desde": val_start,
        "val_hasta": val_end,
        "climatologia_hasta": tr["period_start"].max(),
    })

display(pd.DataFrame(fold_descriptions))

,fold,train_filas,val_filas,train_target_hasta,val_desde,val_hasta,climatologia_hasta
0,F1,1872,432,2004-12-01,2005-01-01,2007-12-01,2004-11-01
1,F2,2304,432,2007-12-01,2008-01-01,2010-12-01,2007-11-01
2,F3,2736,432,2010-12-01,2011-01-01,2013-12-01,2010-11-01
3,F4,3168,576,2013-12-01,2014-01-01,2017-12-01,2013-11-01


## 6. Construcción fold-specific de las características climatológicas

Para cada fold:

1. se obtiene su subentrenamiento;
2. se calcula climatología por `zona + mes` solo hasta el último mes de entrada de ese subentrenamiento;
3. esa referencia transforma tanto el subentrenamiento como su validación;
4. nunca se calcula una media/desviación usando observaciones del bloque de validación.

In [7]:
fold_data = []

for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
    subtrain = train_df[
        train_df["target_period_start"] <= pd.Timestamp(train_end)
    ].copy()

    subval = train_df[
        train_df["target_period_start"].between(
            pd.Timestamp(val_start),
            pd.Timestamp(val_end),
        )
    ].copy()

    cutoff = subtrain["period_start"].max()
    zones = subtrain["zone_id"].unique()

    reference = clim.fit_climatology(
        monthly=monthly,
        cutoff=cutoff,
        zones=zones,
    )

    X_subtrain = clim.add_climate_features(
        frame=subtrain,
        base_X=subtrain[features_v3].copy(),
        reference=reference,
    )
    X_subval = clim.add_climate_features(
        frame=subval,
        base_X=subval[features_v3].copy(),
        reference=reference,
    )

    y_subtrain = label_encoder.transform(subtrain["target_amenaza"])
    y_subval = label_encoder.transform(subval["target_amenaza"])

    assert X_subtrain.shape[1] == 192
    assert X_subval.shape[1] == 192
    assert X_subtrain.isna().sum().sum() == 0
    assert X_subval.isna().sum().sum() == 0

    fold_data.append({
        "name": name,
        "train_frame": subtrain,
        "val_frame": subval,
        "X_train": X_subtrain,
        "y_train": y_subtrain,
        "X_val": X_subval,
        "y_val": y_subval,
        "cutoff": cutoff,
    })

    print(
        f"{name}: train={X_subtrain.shape}, val={X_subval.shape}, "
        f"climatología <= {cutoff.date()}"
    )

F1: train=(1872, 192), val=(432, 192), climatología <= 2004-11-01
F2: train=(2304, 192), val=(432, 192), climatología <= 2007-11-01
F3: train=(2736, 192), val=(432, 192), climatología <= 2010-11-01
F4: train=(3168, 192), val=(576, 192), climatología <= 2013-11-01


## 7. Métricas

In [8]:
def calculate_metrics(y_true, y_pred, class_names=CLASS_NAMES):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        zero_division=0,
    )

    result = {
        "macro_f1": float(macro_f1),
        "balanced_accuracy": float(balanced_acc),
    }

    for i, class_name in enumerate(class_names):
        key = class_name.lower()
        result[f"precision_{key}"] = float(precision[i])
        result[f"recall_{key}"] = float(recall[i])
        result[f"f1_{key}"] = float(f1[i])
        result[f"support_{key}"] = int(support[i])

    return result

## 8. Baselines dentro de los mismos folds

In [9]:
baseline_rows = []

for fold in fold_data:
    Xtr = fold["X_train"]
    ytr = fold["y_train"]
    Xva = fold["X_val"]
    yva = fold["y_val"]

    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(Xtr, ytr)
    dummy_pred = dummy.predict(Xva)
    dummy_metrics = calculate_metrics(yva, dummy_pred)

    persistence_pred = label_encoder.transform(
        fold["val_frame"]["amenaza_mes"]
    )
    persistence_metrics = calculate_metrics(yva, persistence_pred)

    baseline_rows.append({
        "modelo": "Dummy",
        "fold": fold["name"],
        **dummy_metrics,
    })
    baseline_rows.append({
        "modelo": "Persistencia",
        "fold": fold["name"],
        **persistence_metrics,
    })

baseline_fold_table = pd.DataFrame(baseline_rows)

baseline_cv = (
    baseline_fold_table.groupby("modelo")
    .agg(
        macro_f1_cv_mean=("macro_f1", "mean"),
        macro_f1_cv_std=("macro_f1", "std"),
        balanced_accuracy_cv_mean=("balanced_accuracy", "mean"),
        recall_alta_cv_mean=("recall_alta", "mean"),
    )
    .sort_values("macro_f1_cv_mean", ascending=False)
)

display(baseline_cv)

,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean
modelo,,,,
Persistencia,0.369561,0.031969,0.369602,0.349616
Dummy,0.155065,0.016693,0.333333,0.250000


## 9. Cinco modelos y espacios de hiperparámetros

In [10]:
models = {
    "Regresion_Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=4000,
            random_state=RANDOM_STATE,
        )),
    ]),

    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            max_iter=600,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_STATE,
        )),
    ]),
}

param_spaces = {
    "Regresion_Logistica": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
        "model__class_weight": [None, "balanced"],
    },

    "Random_Forest": {
        "n_estimators": [250, 400, 600],
        "max_depth": [None, 10, 18, 26],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5],
        "class_weight": [None, "balanced"],
    },

    "XGBoost": {
        "n_estimators": [200, 350, 500],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.06, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
    },

    "SVM_RBF": {
        "model__C": [0.1, 1.0, 10.0, 50.0],
        "model__gamma": ["scale", 0.001, 0.01, 0.1],
        "model__class_weight": [None, "balanced"],
    },

    "MLP": {
        "model__hidden_layer_sizes": [
            (64,),
            (128, 64),
            (128, 64, 32),
        ],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.0005, 0.001, 0.003],
    },
}

N_ITER = 6

## 10. Búsqueda temporal usando las matrices climatológicas de cada fold

In [11]:
def temporal_random_search_v5(
    model_name,
    estimator,
    param_space,
    folds,
    n_iter=N_ITER,
    random_state=RANDOM_STATE,
):
    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state,
        )
    )

    rows = []

    for config_id, params in enumerate(sampled_params, start=1):
        scores = []
        recalls_alta = []
        started = time.perf_counter()

        for fold in folds:
            candidate = clone(estimator).set_params(**params)

            candidate.fit(
                fold["X_train"],
                fold["y_train"],
            )

            pred = candidate.predict(fold["X_val"])
            metrics = calculate_metrics(fold["y_val"], pred)

            scores.append(metrics["macro_f1"])
            recalls_alta.append(metrics["recall_alta"])

        row = {
            "modelo": model_name,
            "config_id": config_id,
            "macro_f1_cv_mean": float(np.mean(scores)),
            "macro_f1_cv_std": float(np.std(scores)),
            "recall_alta_cv_mean": float(np.mean(recalls_alta)),
            "seconds": time.perf_counter() - started,
            "params": params,
        }
        rows.append(row)

        print(
            f"{model_name} | {config_id:02d}/{len(sampled_params)} | "
            f"Macro F1={row['macro_f1_cv_mean']:.4f} "
            f"± {row['macro_f1_cv_std']:.4f}"
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["macro_f1_cv_mean", "macro_f1_cv_std"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


search_results = {}

for name, estimator in models.items():
    print("\n" + "=" * 80)
    print("AJUSTANDO:", name)

    search_results[name] = temporal_random_search_v5(
        model_name=name,
        estimator=estimator,
        param_space=param_spaces[name],
        folds=fold_data,
    )

search_table = pd.concat(
    search_results.values(),
    ignore_index=True,
)

search_table.to_csv(
    OUTPUT_DIR / "busqueda_hiperparametros_v5.csv",
    index=False,
)

print("\nGuardado:", OUTPUT_DIR / "busqueda_hiperparametros_v5.csv")


AJUSTANDO: Regresion_Logistica
Regresion_Logistica | 01/6 | Macro F1=0.3316 ± 0.0145
Regresion_Logistica | 02/6 | Macro F1=0.3328 ± 0.0144
Regresion_Logistica | 03/6 | Macro F1=0.3357 ± 0.0125
Regresion_Logistica | 04/6 | Macro F1=0.3310 ± 0.0212
Regresion_Logistica | 05/6 | Macro F1=0.3327 ± 0.0112
Regresion_Logistica | 06/6 | Macro F1=0.3267 ± 0.0245

AJUSTANDO: Random_Forest
Random_Forest | 01/6 | Macro F1=0.3376 ± 0.0344
Random_Forest | 02/6 | Macro F1=0.3496 ± 0.0267
Random_Forest | 03/6 | Macro F1=0.3331 ± 0.0245
Random_Forest | 04/6 | Macro F1=0.3352 ± 0.0263
Random_Forest | 05/6 | Macro F1=0.3418 ± 0.0118
Random_Forest | 06/6 | Macro F1=0.3506 ± 0.0143

AJUSTANDO: XGBoost
XGBoost | 01/6 | Macro F1=0.3254 ± 0.0345
XGBoost | 02/6 | Macro F1=0.3455 ± 0.0390
XGBoost | 03/6 | Macro F1=0.3346 ± 0.0387
XGBoost | 04/6 | Macro F1=0.3513 ± 0.0333
XGBoost | 05/6 | Macro F1=0.3213 ± 0.0118
XGBoost | 06/6 | Macro F1=0.3426 ± 0.0404

AJUSTANDO: SVM_RBF
SVM_RBF | 01/6 | Macro F1=0.3323 ± 0.0

## 11. Mejores configuraciones

In [12]:
best_params = {}

for name, result in search_results.items():
    row = result.iloc[0]
    best_params[name] = row["params"]

    print(
        f"{name}: Macro F1 CV={row['macro_f1_cv_mean']:.4f} | "
        f"Recall Alta={row['recall_alta_cv_mean']:.4f} | "
        f"{row['params']}"
    )

with open(
    OUTPUT_DIR / "mejores_hiperparametros_v5.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_params,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

Regresion_Logistica: Macro F1 CV=0.3357 | Recall Alta=0.3568 | {'model__class_weight': 'balanced', 'model__C': 1.0}
Random_Forest: Macro F1 CV=0.3506 | Recall Alta=0.3257 | {'n_estimators': 250, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}
XGBoost: Macro F1 CV=0.3513 | Recall Alta=0.2849 | {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.9}
SVM_RBF: Macro F1 CV=0.3465 | Recall Alta=0.2784 | {'model__gamma': 0.001, 'model__class_weight': None, 'model__C': 1.0}
MLP: Macro F1 CV=0.3386 | Recall Alta=0.3456 | {'model__learning_rate_init': 0.0005, 'model__hidden_layer_sizes': (128, 64), 'model__alpha': 0.01}


## 12. Comparación interna V5

Esta es la única tabla que se debe utilizar por ahora para decidir si la
climatología ERA5-Land merece pasar a la validación 2018–2021.

In [13]:
model_rows = []

for name, result in search_results.items():
    best = result.iloc[0]
    model_rows.append({
        "modelo": name,
        "macro_f1_cv_mean": best["macro_f1_cv_mean"],
        "macro_f1_cv_std": best["macro_f1_cv_std"],
        "recall_alta_cv_mean": best["recall_alta_cv_mean"],
    })

model_cv = pd.DataFrame(model_rows)

baseline_for_display = (
    baseline_cv.reset_index()[
        [
            "modelo",
            "macro_f1_cv_mean",
            "macro_f1_cv_std",
            "recall_alta_cv_mean",
        ]
    ]
)

comparison = (
    pd.concat(
        [model_cv, baseline_for_display],
        ignore_index=True,
    )
    .sort_values("macro_f1_cv_mean", ascending=False)
    .reset_index(drop=True)
)

display(comparison)

comparison.to_csv(
    OUTPUT_DIR / "comparacion_cv_interna_v5.csv",
    index=False,
)

print(
    "\nNO ejecutar todavía 2018–2021, prueba temporal ni holdout espacial."
)

,modelo,macro_f1_cv_mean,macro_f1_cv_std,recall_alta_cv_mean
0,Persistencia,0.369561,0.031969,0.349616
1,XGBoost,0.351277,0.033303,0.284860
2,Random_Forest,0.350620,0.014301,0.325725
3,SVM_RBF,0.346506,0.024771,0.278440
4,MLP,0.338562,0.028303,0.345562
5,Regresion_Logistica,0.335689,0.012542,0.356758
6,Dummy,0.155065,0.016693,0.250000



NO ejecutar todavía 2018–2021, prueba temporal ni holdout espacial.
